# Bangla Distill v2 — Retrained Student Model

**Fixes applied in v2:**
- ✅ **Base Model**: `Qwen/Qwen2.5-0.5B-Instruct` (was: raw base model with no instruction-following)
- ✅ **Dataset Format**: ChatML `messages` pairs (was: raw text monologues)
- ✅ **Loss Masking**: Assistant-only loss via `completion_only_loss` (was: full-sequence CLM)
- ✅ **Training Duration**: 3 epochs with early stopping (was: 1 epoch, 375 steps)
- ✅ **Validation**: 10% held-out validation with eval every 50 steps (was: none)
- ✅ **System Prompt**: Bengali-language system prompt baked into training

**Unchanged (already good):**
- LoRA rank=16, alpha=32, all 7 projection targets
- 4-bit NF4 quantization via BitsAndBytes
- Cosine LR schedule with warmup

> **Runtime Requirements:** Kaggle P100 / T4 GPU, or Colab T4.
> Make sure GPU is enabled: Runtime > Change runtime type > GPU

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install -q -U "bitsandbytes>=0.46.1" transformers==4.51.3 peft==0.20.0 trl==1.12.0 accelerate bitsandbytes sentencepiece datasets


## Step 2: Upload Dataset Files

Upload the two files generated by `prepare_chat_dataset.py`:
- `train_chat.jsonl` (~7.5 MB, 5,387 records)
- `val_chat.jsonl` (~829 KB, 598 records)

In [ ]:
import os

TRAIN_FILE = None
VAL_FILE = None

# 1. Search for dataset files across potential locations (current dir, /content, /kaggle/input)
search_roots = [".", "/content", "/kaggle/input"]
for root_dir in search_roots:
    if os.path.exists(root_dir):
        for root, dirs, files in os.walk(root_dir):
            if "train_chat.jsonl" in files:
                TRAIN_FILE = os.path.join(root, "train_chat.jsonl")
            if "val_chat.jsonl" in files:
                VAL_FILE = os.path.join(root, "val_chat.jsonl")
            if TRAIN_FILE and VAL_FILE:
                print(f"Found dataset in: {root}")
                break
        if TRAIN_FILE and VAL_FILE:
            break

# 2. If running in Google Colab and files are not yet uploaded, trigger browser upload
if not (TRAIN_FILE and os.path.exists(TRAIN_FILE)):
    try:
        from google.colab import files
        print("Dataset not found locally. Please upload train_chat.jsonl and val_chat.jsonl:")
        uploaded = files.upload()
        for fname in ["train_chat.jsonl", "val_chat.jsonl"]:
            if os.path.exists(fname):
                if fname == "train_chat.jsonl":
                    TRAIN_FILE = os.path.abspath(fname)
                else:
                    VAL_FILE = os.path.abspath(fname)
    except ImportError:
        pass

# 3. Print status and verify files
print(f"\nTrain file: {TRAIN_FILE}")
print(f"Val file:   {VAL_FILE}")

all_found = True
for name, fp in [("Train", TRAIN_FILE), ("Val", VAL_FILE)]:
    if fp and os.path.exists(fp):
        size_kb = os.path.getsize(fp) / 1024
        lines = sum(1 for _ in open(fp, encoding='utf-8'))
        print(f"  ✅ {name} ({fp}): {lines} records ({size_kb:.1f} KB)")
    else:
        print(f"  ❌ {name}: NOT FOUND")
        all_found = False

if not all_found:
    print("\n" + "="*60)
    print("⚠️  HOW TO FIX MISSING DATASET FILES:")
    print("="*60)
    if os.path.exists("/kaggle"):
        print("👉 ON KAGGLE:")
        print("   1. Look at the right sidebar under 'Input' -> click '+ Add Input' / 'Upload Dataset'.")
        print("   2. Upload train_chat.jsonl and val_chat.jsonl from your computer (D:\\Bangla Distill\\).")
        print("   3. After creating the dataset, re-run this cell.")
    else:
        print("👉 ON GOOGLE COLAB:")
        print("   1. Click the Folder icon (📁) on the left sidebar.")
        print("   2. Drag & drop train_chat.jsonl and val_chat.jsonl into the files area.")
        print("   3. Once uploaded, re-run this cell.")
    print("="*60)


## Step 3: Load Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": TRAIN_FILE,
    "validation": VAL_FILE,
})

print(dataset)
print(f"\nSample record:")
print(dataset["train"][0])

## Step 4: Configure Base Model & QLoRA

**Key change:** Using `Qwen/Qwen2.5-0.5B-Instruct` instead of the raw base model.
The Instruct variant already understands ChatML special tokens (`<|im_start|>`, `<|im_end|>`).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

# ---- v2 FIX: Use the INSTRUCT model, not the raw base ----
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

OUTPUT_DIR = "/kaggle/working/qwen-2.5-0.5b-bangla-student-v2" if os.path.exists("/kaggle") else "/content/qwen-2.5-0.5b-bangla-student-v2"

# GPU diagnostics
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    raise RuntimeError("CUDA GPU required! Enable GPU in Runtime settings.")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded: {len(tokenizer)} tokens")
print(f"Chat template present: {tokenizer.chat_template is not None}")

# Load base model: try 4-bit NF4; if bitsandbytes issue, fallback to native FP16
try:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    print("Base model loaded (4-bit NF4 quantized)")
    USING_4BIT = True
except Exception as e:
    print(f"Note: 4-bit quantization skipped ({e}). Loading in native FP16 (model is only ~1 GB, fits easily in GPU)...\n")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Base model loaded (native FP16)")
    USING_4BIT = False


## Step 5: Configure LoRA Adapter

In [ ]:
# Bypass incompatible torchao checks across all PEFT modules (torchao is optional and not needed for LoRA)
import peft.import_utils
import peft.tuners.lora.torchao
peft.import_utils.is_torchao_available = lambda: False
peft.tuners.lora.torchao.is_torchao_available = lambda: False
try:
    import peft.utils.quantization_utils
    peft.utils.quantization_utils.is_torchao_available = lambda: False
except Exception:
    pass

from peft import LoraConfig, get_peft_model

# LoRA config (same as v1 — these settings were already good)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Step 6: Train with SFTTrainer (v2 Fixes Applied)

**Key fixes:**
1. Uses native `messages` format — SFTTrainer auto-applies `apply_chat_template`
2. `completion_only_loss=True` — only computes loss on assistant response tokens
3. 3 epochs instead of 1
4. Validation every 50 steps with early stopping

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

sft_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_steps": 25,
    "max_grad_norm": 0.3,
    "optim": "paged_adamw_8bit" if USING_4BIT else "adamw_torch",
    "weight_decay": 0.0,
    "fp16": True,
    "bf16": False,
    "eval_strategy": "steps",
    "eval_steps": 50,
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "save_strategy": "steps",
    "save_steps": 50,
    "save_total_limit": 5,
    "logging_steps": 10,
    "report_to": [],
    "gradient_checkpointing": True,
    "seed": 42,
}

# Handle version differences across TRL dynamically
sig_params = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sig_params:
    sft_kwargs["max_length"] = 1024
elif "max_seq_length" in sig_params:
    sft_kwargs["max_seq_length"] = 1024

if "assistant_only_loss" in sig_params:
    sft_kwargs["assistant_only_loss"] = True

training_args = SFTConfig(**sft_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": dataset["train"],
    "eval_dataset": dataset["validation"],
}
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print(f"Training config:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Train records: {len(dataset['train'])}")
print(f"  Val records: {len(dataset['validation'])}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Eval every {training_args.eval_steps} steps")


In [ ]:
# Start training!
print("Starting training...")
trainer.train()
print("Training complete!")

## Step 7: Save the Final LoRA Adapter

In [ ]:
import json

FINAL_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final-lora-adapter-v2")
os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

# Save LoRA adapter weights
model.save_pretrained(FINAL_ADAPTER_DIR)
print(f"LoRA adapter saved to: {FINAL_ADAPTER_DIR}")

# Save tokenizer (includes chat template)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)
print(f"Tokenizer saved.")

# Save training info
training_info = f"""Qwen2.5-0.5B-Instruct Bengali Fine-Tuning (v2)
=============================================

Base model: {BASE_MODEL}
Dataset: ChatML instruction-response pairs
Training examples: {len(dataset['train'])}
Validation examples: {len(dataset['validation'])}
Method: QLoRA (4-bit NF4)
LoRA rank: 16
LoRA alpha: 32
Epochs: {int(training_args.num_train_epochs)}
Loss: Completion-only (assistant tokens only)
Compute: FP16

Key v2 improvements:
- Instruct base model (ChatML-aware)
- Instruction-response dataset format
- Assistant-only loss masking
- 3 epochs with validation early stopping
"""

with open(os.path.join(FINAL_ADAPTER_DIR, "TRAINING_INFO.txt"), "w") as f:
    f.write(training_info)

# List output files
print(f"\nFinal adapter contents:")
for fn in sorted(os.listdir(FINAL_ADAPTER_DIR)):
    size = os.path.getsize(os.path.join(FINAL_ADAPTER_DIR, fn))
    print(f"  {fn:45s} {size/1024:.1f} KB")

## Step 8: Quick Validation — Test the Retrained Model

In [ ]:
def generate_test(prompt, max_new_tokens=200, temperature=0.3, top_p=0.8):
    """Generate a response using the retrained model."""
    messages = [
        {"role": "system", "content": "আপনি একজন বাংলা ভাষায় দক্ষ সহকারী। ব্যবহারকারীর প্রশ্নের উত্তর সঠিক তথ্যসহ বিস্তারিতভাবে বাংলায় দিন।"},
        {"role": "user", "content": prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=max(0.01, temperature),
            top_p=top_p,
            repetition_penalty=1.2,
            do_sample=temperature > 0.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()

test_prompts = [
    ("History", "বাংলাদেশের ইতিহাস ও স্বাধীনতা যুদ্ধ সম্পর্কে বলুন।"),
    ("AI", "কৃত্রিম বুদ্ধিমত্তা (Artificial Intelligence) কী এবং এটি কীভাবে কাজ করে?"),
    ("Nature", "সুন্দরবনের গুরুত্ব এবং এর রয়েল বেঙ্গল টাইগার সম্পর্কে বিস্তারিত লিখুন।"),
    ("Health", "একজন শিক্ষার্থীর সুস্থ ও কর্মক্ষম থাকার জন্য তিনটি প্রয়োজনীয় নিয়ম কী?"),
]

print("=" * 60)
print("v2 RETRAINED MODEL — VALIDATION TESTS")
print("=" * 60)
for category, prompt in test_prompts:
    print(f"\n--- {category} ---")
    print(f"Prompt: {prompt}")
    resp = generate_test(prompt)
    print(f"Response: {resp}")
    print("-" * 60)

## Step 9: Download the Adapter

Download the `final-lora-adapter-v2/` folder to replace your local adapter.

In [ ]:
import shutil

# Create a zip for easy download
zip_path = shutil.make_archive(
    os.path.join(OUTPUT_DIR, "final-lora-adapter-v2"),
    "zip",
    FINAL_ADAPTER_DIR
)
print(f"Zip created: {zip_path} ({os.path.getsize(zip_path) / (1024**2):.1f} MB)")

# Auto-download on Colab
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Download manually from: {zip_path}")